In [ ]:
from src.model import SPDMatrixLearner

from src.utils import encode_df
from src.compute_text_representations import compute_text_representations
from src.pairwise_dataset import PairwiseDataset
from src.train import train
from src.sampling_theory import estimate_corrs
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.templates.default = "simple_white"
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns
import numpy as np
import torch
from itertools import combinations

torch.set_float32_matmul_precision("medium")
from time import time
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import logging
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm
import sys
from src.feature_importance import feature_importance
from loguru import logger

In [ ]:
logger.remove()
logger.add(sys.stderr, level="INFO")

In [ ]:
device = "cuda"
df = pd.read_csv("datasets/relative_clause.csv")
sentences = df["sentence"].tolist()
df = df.drop(columns="sentence")
embeddings = compute_text_representations(
    sentences, model_name="bert-base-uncased", token_aggregation="mean"
)
X = encode_df(df).to(device)
Y = embeddings[5].to(device)
# X_train, X_test, Y_train, Y_test = train_test_split(
#     X, Y, test_size=0.2, random_state=0
# )

In [ ]:
corrs, n_pairs = estimate_corrs(
    X, bootstrap=5, init_sample_size=32, max_margin=0.01
)

In [ ]:
n_pairs = 40 * X.shape[1] ** 2
n_pairs

In [ ]:
all_importances = []
all_spearman = []
for i in tqdm(range(len(embeddings))):
    Y = embeddings[i].to(device)
    model = SPDMatrixLearner(X.shape[1])
    dataset = PairwiseDataset(X, Y, n_pairs=n_pairs, gamma=1)
    dataloader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        collate_fn=lambda x: x[0],
    )
    model = train(model, dataloader, device=device)
    importances, spearman = feature_importance(
        model,
        dataloader,
        df.columns.values,
        n_perm=30,
        alpha=0.01,
        warn_ci=0.01,
    )
    importances["layer"] = i
    all_importances.append(importances)
    spearman["layer"] = i
    all_spearman.append(spearman)
all_importances = pd.concat(all_importances)
all_spearman = pd.DataFrame(all_spearman)

In [ ]:
px.line(
    all_importances.query("mean > 0.1"),
    x="layer",
    y="mean",
    color="Feature",
    error_y="std",
    height=800,
)

In [ ]:
px.line(all_spearman, x="layer", y="mean", error_y="std")

In [ ]:
model = SPDMatrixLearner(X.shape[1])

In [ ]:
dataset = PairwiseDataset(X, Y, n_pairs=n_pairs, gamma=1)
dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=lambda x: x[0],
)

In [ ]:
model = train(model, dataloader, device=device)

In [ ]:
importances, spearman = feature_importance(
    model, dataloader, df.columns.values, n_perm=30, alpha=0.01, warn_ci=0.01
)

In [ ]:
px.bar(
    importances.query("mean > 0.05"),
    x="mean",
    y="Feature",
    color="Feature",
    error_x="std",
    orientation="h",
)

In [ ]:
from captum.attr import FeaturePermutation

In [ ]:
from joblib import Parallel, delayed
from tqdm.auto import tqdm

In [ ]:
importances = []
perf = []
for _ in tqdm(range(5)):
    X_batch, Y_batch = dataset.sample(4096)
    X_batch_flat = (X_batch[:, None] * X_batch[:, :, None])[
        :, *model.triu_indices
    ]

    def f(x):
        pred = model.flat_forward(x)
        return model.spearman(pred, Y_batch)

    feature_perm = FeaturePermutation(f)
    batch_importances = feature_perm.attribute(X_batch_flat)
    batch_importances = batch_importances.cpu().squeeze()
    importances.append(batch_importances)
    perf.append(model.spearman(model(X_batch), Y_batch).item())
importances = torch.stack(importances)

In [ ]:
px.bar(
    importances,
    x="mean",
    y="Feature",
    color="Feature",
    orientation="h",
    error_x_minus="lower_ci",
    error_x="upper_ci",
)

In [ ]:
corrs = pd.DataFrame(corrs.cpu(), columns=features, index=features)

In [ ]:
px.imshow(
    corrs,
    height=800,
    color_continuous_midpoint=0,
    color_continuous_scale="RdBu",
)

In [ ]:
px.imshow(
    W, color_continuous_midpoint=0, color_continuous_scale="RdBu", height=800
)

In [ ]:
mean_importances = torch.full(model.get_W().shape, torch.nan)
mean_importances[*model.triu_indices] = importances.mean(dim=0)
mean_importances = pd.DataFrame(
    mean_importances.T, columns=features, index=features
)

In [ ]:
px.imshow(mean_importances, height=800)

In [ ]:
X_batch, Y_batch = dataset.sample(n_pairs)

In [ ]:
model.spearman(model(X_batch), Y_batch)

In [ ]:
X_batch_flat = (X_batch[:, None] * X_batch[:, :, None])[:, *model.triu_indices]
X_batch_flat.shape

In [ ]:
feature_perm.attribute(X_batch_flat)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

In [ ]:
proj = PCA(n_components=0.95).fit_transform(Y.cpu().numpy())

In [ ]:
proj.shape

In [ ]:
# df[["x", "y"]] = PCA(n_components=2).fit_transform(Y.cpu().numpy())
df[["x", "y"]] = TSNE(n_components=2).fit_transform(proj)

In [ ]:
# df[["x", "y"]] = PCA(n_components=2).fit_transform(Y.cpu().numpy())
df[["x", "y"]] = Y[:, [0, 1]].cpu().numpy()

In [ ]:
color = df.sentence_CLAUSE

In [ ]:
color = df.obj_NUM

In [ ]:
color = df.intervener_NUM

In [ ]:
color = df.sentence_RC_attached

In [ ]:
color = df.has_embed.astype(str) + ", " + df.sentence_length.astype(str)

In [ ]:
color = df.obj_ZIPF

In [ ]:
color = df.sentence_length

In [ ]:
px.scatter(
    df,
    x="x",
    y="y",
    color=color,
    hover_name=sentences,
    hover_data=df.columns,
    height=800,
)